<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

#### 1. Pembersihan data dan perhitungan fitur density

In [11]:
# Memilih dataset dan parameter pemisahan
import pandas as pd
from IPython.display import display, Markdown
from pci_ml.data import list_datasets, load_data, clean_data, split_data
from pci_ml.memory import configure, release
from pci_ml.ui import build_panel

EXPORT_MODE = True   # True = print panel values instead of showing widgets
RELEASE_MEMORY = True

DEFAULTS = {
    "file": None,        # None = first dataset, or e.g. "data_2026.xlsx"
    "n_seed": 200,
    "test_size": 0.2,
    "share_tol": 0.01,
    "pci_min": 0,
}

configure(release=RELEASE_MEMORY)

datasets = list_datasets()
if DEFAULTS["file"] is not None and DEFAULTS["file"] not in datasets:
    raise ValueError(f"DEFAULTS['file'] '{DEFAULTS['file']}' not found in datasets")

# Helper function: compact preview of the first rows
def format_adaptive_df(df_input):
    if df_input is None:
        return ""
    df_head = df_input.head()
    left_count = 5
    right_count = 2
    if df_head.shape[1] > (left_count + right_count):
        left_cols = df_head.iloc[:, :left_count]
        right_cols = df_head.iloc[:, -right_count:]
        middle_col = pd.DataFrame({"…": ["…"] * len(df_head)}, index=df_head.index)
        df_display = pd.concat([left_cols, middle_col, right_cols], axis=1)
    else:
        df_display = df_head
    markdown_table = df_display.to_markdown(index=False)
    return f'<div style="font-size: 11px;">\n\n{markdown_table}\n\n</div>'

df = None
handle = None

def on_select(name):
    raw = None if name is None else load_data(name)
    handle.update(Markdown(format_adaptive_df(raw)))

panel = build_panel(datasets, DEFAULTS, on_file_change=on_select, export=EXPORT_MODE)

handle = display(Markdown(""), display_id=True)

on_select(panel.values()["file"])


File terpilih untuk analisis: pci-ml-26.xlsx
Parameter Pemisahan Data:
- Jumlah seed     : 200
- Test size       : 0.2
- Share tolerance : 0.01
- PCI minimum     : 0.0


<div style="font-size: 11px;">

| ROAD_NAME    | ROAD_ID   | SEGMENT_ID   |   SEGMENT_AREA |   GATOR_CRACK_L | …   |   RAVELLING_H |     PCI |
|:-------------|:----------|:-------------|---------------:|----------------:|:----|--------------:|--------:|
| Guru Bangkol | R001      | S001         |            450 |            0    | …   |             0 | 85.4969 |
| Guru Bangkol | R001      | S002         |            450 |            0    | …   |             0 | 73.3368 |
| Guru Bangkol | R001      | S003         |            450 |            0    | …   |             0 | 89.9305 |
| Guru Bangkol | R001      | S004         |            450 |            0.64 | …   |             0 | 73.6287 |
| Guru Bangkol | R001      | S005         |            450 |            0    | …   |             0 | 96.1874 |

</div>

In [12]:
# Membersihkan data dari duplikat, pencilan dan data tidak valid, serta menghitung fitur density
cfg = panel.values()
filename = cfg["file"]
df, features = clean_data(load_data(filename))

if not EXPORT_MODE:
    print(f"Parameter: file {filename} | jumlah seed {cfg['n_seed']} | "
          f"test size {cfg['test_size']} | toleransi {cfg['share_tol']} | "
          f"PCI minimum {cfg['pci_min']}")

markdown_table = format_adaptive_df(df)
print("Data telah dibersihkan dan fitur densitas telah dihitung.")
display(Markdown(markdown_table))


Data telah dibersihkan dan fitur densitas telah dihitung.


<div style="font-size: 11px;">

| ROAD_NAME    | ROAD_ID   | SEGMENT_ID   |   GATOR_CRACK_L |   GATOR_CRACK_M | …   |   RAVELLING_H |     PCI |
|:-------------|:----------|:-------------|----------------:|----------------:|:----|--------------:|--------:|
| Guru Bangkol | R001      | S001         |      0          |      0          | …   |             0 | 85.4969 |
| Guru Bangkol | R001      | S002         |      0          |      0.00391778 | …   |             0 | 73.3368 |
| Guru Bangkol | R001      | S003         |      0          |      0          | …   |             0 | 89.9305 |
| Guru Bangkol | R001      | S004         |      0.00142222 |      0.0231111  | …   |             0 | 73.6287 |
| Guru Bangkol | R001      | S005         |      0          |      0          | …   |             0 | 96.1874 |

</div>

#### 2. Pemisahan data latih dan data uji

In [13]:
# Memisah data sesuai kriteria
from IPython.display import clear_output

if globals().get("df") is None:
    df, features = clean_data(load_data(filename))

def show_progress(done, total, best_gap):
    if done % 100 != 0 and done != total:
        return
    gap = "-" if best_gap is None else f"{best_gap:.4f}"
    clear_output(wait=True)
    print(f"Mencari Seeds: {done}/{total} | Best Gap: {gap}", flush=True)

result = split_data(df, features, filename, {
    "n_seed": cfg["n_seed"],
    "test_size": cfg["test_size"],
    "share_tol": cfg["share_tol"],
    "pci_min": cfg["pci_min"],
}, progress=show_progress)
print()

split_summary = result["summary"]

# Print the summary outside the progress line so it always shows up
print(split_summary)

release("result", "df", "show_progress")


Mencari Seeds: 200/200 | Best Gap: 0.7556

Seed used: 179 (Gap: 0.7556)
Train: 933 rows (79.27%) | Test: 244 rows (20.73%)
Roads: 27 train | 7 test
Saved to: "C:\Users\gearb\Desktop\pci-ml-single\pci-ml\datasets\pci-ml-26\n200_t0.2_s0.01_p0.0"


#### 3. Pelatihan model *machine learning*

In [14]:
# Mempersiapkan input model
import numpy as np
from pci_ml.data import PCI, ROAD_ID, METADATA, latest_checkpoint, load_checkpoint

SEED = 42   # model seed (RF, XGBoost, Keras), separate from the split seed in meta["seed"]

# Memuat checkpoint terbaru untuk file yang dipilih
ckpt_tag = latest_checkpoint(filename)

X_train = X_test = y_train = y_test = groups_train = test_ids = meta = None

def load_latest_ckpt(tag):
    global X_train, X_test, y_train, y_test, groups_train, test_ids, features, meta
    # train_df / test_df are local: they are freed when this function returns
    train_df, test_df, features, meta = load_checkpoint(filename, tag)

    X_train = train_df[features].to_numpy(dtype="float32")
    X_test  = test_df[features].to_numpy(dtype="float32")
    y_train = train_df[PCI].to_numpy(dtype="float32")    # PCI scale 0-100; divide by 100 in the ANN cell
    y_test  = test_df[PCI].to_numpy(dtype="float32")

    groups_train = train_df[ROAD_ID].to_numpy()          # for GroupKFold when tuning
    test_ids = test_df[[ROAD_ID, *METADATA]].reset_index(drop=True)   # labels for result tables

    assert all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)), "NaN/inf found"

def split_info():
    n_tr, n_te = len(X_train), len(X_test)
    total = n_tr + n_te
    return (f"**Train:** {X_train.shape} ({n_tr / total:.2%}) | "
            f"**Test:** {X_test.shape} ({n_te / total:.2%})")

load_latest_ckpt(ckpt_tag)

# Staleness guard: the newest split may have been made with different parameters
current = {key: cfg[key] for key in ("n_seed", "test_size", "share_tol", "pci_min")}
if current != meta["params"]:
    print("PERINGATAN: split terbaru dibuat dengan parameter berbeda. "
          "Jalankan ulang Cell 7 untuk memisahkan data dengan parameter saat ini.")
    print(f"  Parameter saat ini : {current}")
    print(f"  Parameter split    : {meta['params']}")

display(Markdown(f"**Split terpilih:** `{ckpt_tag}`\n\n{split_info()}"))

release("df", "ckpt_tag", "load_latest_ckpt", "split_info", "current")


**Split terpilih:** `n200_t0.2_s0.01_p0.0`

**Train:** (933, 55) (79.27%) | **Test:** (244, 55) (20.73%)

In [15]:
# DINONAKTIFKAN SEMENTARA: verifikasi data input
# # Verifikasi data input
# import numpy as np
# from pci_ml.data import ROAD_ID

# # 1. Shapes and dtypes
# print("Arrays:")
# for name in ("X_train", "X_test", "y_train", "y_test", "groups_train"):
#     a = globals()[name]
#     print(f"  {name:<14}{str(a.shape):<12}{a.dtype}")
# print(f"  {'test_ids':<14}{str(test_ids.shape):<12}columns: {list(test_ids.columns)}")
# print(f"  {'features':<14}{len(features)} columns | first 3: {features[:3]}")

# # 2. Consistency checks
# checks = {
#     "X columns == len(features)":        X_train.shape[1] == X_test.shape[1] == len(features),
#     "X/y/groups rows match (train)":     len(X_train) == len(y_train) == len(groups_train),
#     "X/y/test_ids rows match (test)":    len(X_test) == len(y_test) == len(test_ids),
#     "row counts match meta":             (len(X_train), len(X_test)) == (meta["n_train"], meta["n_test"]),
#     "no NaN/inf in arrays":              all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)),
#     "X within 0-1":                      min(X_train.min(), X_test.min()) >= 0 and max(X_train.max(), X_test.max()) <= 1,
#     "y within 0-100":                    min(y_train.min(), y_test.min()) >= 0 and max(y_train.max(), y_test.max()) <= 100,
#     "no road shared by train and test":  set(groups_train).isdisjoint(test_ids[ROAD_ID]),
#     "train_df/test_df released":         "train_df" not in globals() and "test_df" not in globals(),
# }
# print("\nChecks:")
# for label, ok in checks.items():
#     print(f"  [{'OK' if ok else 'FAIL'}] {label}")

# # 3. Summary of the selected split
# print(f"\nSeed split: {meta['seed']} | Gap: {meta['gap']:.4f} | Dibuat: {meta['created_at']}")
# print(f"PCI train: mean {y_train.mean():.2f}, std {y_train.std(ddof=1):.2f} | "
#       f"test: mean {y_test.mean():.2f}, std {y_test.std(ddof=1):.2f}")

In [16]:
# DINONAKTIFKAN SEMENTARA: baseline Random Forest.
# Cell ini tidak dipakai oleh tuning maupun model final (lihat cell tuning).
# Hapus tanda '#' di awal setiap baris untuk mengaktifkan kembali.

# # Melatih dan menguji model Random Forest
# from pci_ml.random_forest import train_random_forest, evaluate_random_forest
# from pci_ml.ui import build_param_panel
#
# RF_DEFAULTS = {
#     "n_estimators": 300,
#     "max_depth": 0,          # 0 = kedalaman tidak dibatasi
#     "min_samples_split": 2,
#     "min_samples_leaf": 1,
#     "max_features": 1.0,
#     "max_samples": 1.0,      # 1.0 = semua sampel
# }
#
# RF_SPECS = {
#     "n_estimators":      {"type": "int",   "description": "Jumlah pohon:", "label": "Jumlah pohon"},
#     "max_depth":         {"type": "int",   "description": "Kedalaman maks:", "label": "Kedalaman maks"},
#     "min_samples_split": {"type": "int",   "description": "Min sampel split:", "label": "Min sampel split"},
#     "min_samples_leaf":  {"type": "int",   "description": "Min sampel daun:", "label": "Min sampel daun"},
#     "max_features":      {"type": "float", "description": "Maks fitur:", "label": "Maks fitur"},
#     "max_samples":       {"type": "float", "description": "Maks sampel:", "label": "Maks sampel"},
# }
#
# rf_panel = build_param_panel(RF_DEFAULTS, RF_SPECS, "Parameter Random Forest:", export=EXPORT_MODE)
# rf_params = rf_panel.values()
#
# if rf_params["max_depth"] == 0:
#     print("Catatan: kedalaman 0 berarti tanpa batas.")
#
# model = train_random_forest(X_train, y_train, rf_params, random_state=SEED)
# train_metrics = evaluate_random_forest(model, X_train, y_train)
# test_metrics = evaluate_random_forest(model, X_test, y_test)
#
# print("\nHasil Random Forest:")
# print(f"- Data latih: MAE {train_metrics['mae']:.4f} | RMSE {train_metrics['rmse']:.4f} | R² {train_metrics['r2']:.4f}")
# print(f"- Data uji  : MAE {test_metrics['mae']:.4f} | RMSE {test_metrics['rmse']:.4f} | R² {test_metrics['r2']:.4f}")

In [ ]:
# Tuning hyperparameter Random Forest dengan GroupKFold
from IPython.display import Markdown, display, HTML
from pci_ml.random_forest import tune_random_forest, evaluate_random_forest
from pci_ml.ui import build_param_panel

TUNING_DEFAULTS = {
    "n_iter": 100,
    "cv_splits": 5,
}

TUNING_SPECS = {
    "n_iter":    {"type": "int", "description": "Jumlah kandidat:", "label": "Jumlah kandidat"},
    "cv_splits": {"type": "int", "description": "Jumlah fold:", "label": "Jumlah fold"},
}

RF_SEARCH_SPACE = {
    "n_estimators": [300, 500],
    "max_depth": [None, 40, 30, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 3],
    "max_features": [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0],
    "max_samples": [None, 0.6, 0.7, 0.8, 0.9],
}

tuning_panel = build_param_panel(
    TUNING_DEFAULTS, TUNING_SPECS, "Parameter Tuning Random Forest:", export=EXPORT_MODE
)
tuning_params = tuning_panel.values()

display(Markdown(
    f"**Memulai tuning:** {tuning_params['n_iter']} kandidat | "
    f"{tuning_params['cv_splits']} fold CV"
))
rf_result = tune_random_forest(
    X_train, y_train, groups_train,
    RF_SEARCH_SPACE,
    n_iter=tuning_params["n_iter"],
    cv_splits=tuning_params["cv_splits"],
    random_state=SEED,
)

rf_best_params = rf_result["params"]
rf_cv = rf_result["cv_pooled"]
tuned_model = rf_result["model"]
tuned_test = evaluate_random_forest(tuned_model, X_test, y_test)

def format_value(value):
    return "tanpa batas" if value is None else value

param_explanations = {
    "n_estimators": "Lebih banyak pohon = prediksi lebih stabil, tapi lebih lambat",
    "max_depth": "Membatasi kedalaman pohon untuk mencegah overfitting",
    "min_samples_split": "Node butuh minimal ini sampel sebelum bisa dipecah",
    "min_samples_leaf": "Setiap daun minimal punya ini sampel",
    "max_features": "Fitur acak dipilih di tiap split; kurangi untuk dekorelasi pohon",
    "max_samples": "Fraksi data latih per pohon (bootstrap); <1.0 = subsampling",
}

rows_html = ""
for key in RF_SEARCH_SPACE:
    val = format_value(rf_best_params[key])
    desc = param_explanations[key]
    rows_html += (
        "    <tr style='border-bottom: 1px solid #eee;'>\n"
        f"      <td style='padding: 6px 10px; font-family: monospace; white-space: nowrap;'>{key}</td>\n"
        f"      <td style='padding: 6px 10px; text-align: center;'>{val}</td>\n"
        f"      <td style='padding: 6px 10px; max-width: 400px; word-wrap: break-word;'>{desc}</td>\n"
        "    </tr>\n"
    )

param_table = (
    "<table style='font-size: 11px; border-collapse: collapse; width: 100%;'>\n"
    "  <thead>\n"
    "    <tr style='border-bottom: 2px solid #ddd;'>\n"
    "      <th style='padding: 6px 10px; text-align: left; white-space: nowrap;'>Hyperparameter</th>\n"
    "      <th style='padding: 6px 10px; text-align: center; white-space: nowrap;'>Nilai</th>\n"
    "      <th style='padding: 6px 10px; text-align: left;'>Keterangan</th>\n"
    "    </tr>\n"
    "  </thead>\n"
    "  <tbody>\n"
    + rows_html
    + "  </tbody>\n"
    "</table>"
)

metric_table = (
    "| Evaluasi | MAE | RMSE | R\u00b2 |\n"
    "|---|---:|---:|---:|\n"
    f"| CV pooled OOF | {rf_cv['mae']:.4f} | {rf_cv['rmse']:.4f} | {rf_cv['r2']:.4f} |\n"
    f"| Test | {tuned_test['mae']:.4f} | {tuned_test['rmse']:.4f} | {tuned_test['r2']:.4f} |\n"
)

n_test_roads = test_ids["ROAD_ID"].nunique()
explanation = f"""
**Keterangan**
- **MAE**: rata-rata selisih absolut antara prediksi dan PCI aktual, dalam satuan poin PCI. Semakin kecil semakin baik.
- **RMSE**: akar rata-rata kuadrat error, dalam satuan poin PCI. Lebih sensitif terhadap error besar; semakin kecil semakin baik.
- **R\u00b2**: proporsi variasi PCI yang dijelaskan model. Semakin mendekati 1 semakin baik.
- **CV pooled OOF**: semua prediksi out-of-fold dari {tuning_params['cv_splits']} fold digabung, lalu dihitung sekali. Ini estimasi generalisasi yang lebih stabil daripada rata-rata R\u00b2 per fold, tetapi sedikit optimistis karena hyperparameter dipilih dari CV yang sama.
- **Test**: evaluasi pada data uji yang tidak dipakai saat tuning. Ini estimasi akhir pada {n_test_roads} road holdout, tetapi hanya satu holdout sehingga variansnya lebih tinggi.
"""

display(Markdown("### Hyperparameter terbaik"))
display(HTML(param_table))
display(Markdown(f"### Hasil evaluasi\n\n{metric_table}\n\n{explanation}"))


### ***Source code* modul dan fungsi**

In [18]:
# import os
# from pathlib import Path
# from IPython.display import display, Markdown

# # Path to your src directory
# src_dir = Path("src")

# # Loop through all Python files and render them as Markdown code blocks
# for filepath in src_dir.rglob("*.py"): # rglob searches subdirectories too
    
#     # Ignore any files inside Jupyter checkpoint folders
#     if ".ipynb_checkpoints" in filepath.parts:
#         continue

#     with open(filepath, "r", encoding="utf-8") as f:
#         code = f.read()
    
#     # Creates a nice header for the filename and formats the code
#     display(Markdown(f"#### `{filepath.name}`\n```python\n{code}\n```\n---"))